# 05 · Cheat-sheet + Expression/Validation Plan

**Standard slot:** *validation plan.* **For Project 02 this means:** turn the Pareto map (nb 04) into
the **MPNN settings cheat-sheet** (D★) and a concrete expression + validation plan with controls
(D4/D5). Also the recommendation heuristic and the MPNNsol surface-redesign stretch task.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · The MPNN settings cheat-sheet (D★)

Read the recommendations off your Pareto front (nb 04), by goal. Fill the `?`s from
`results/settings_summary.csv` — this card is the deliverable later cohort projects actually use.
Save it to the repo root / `data/CHEATSHEET.md`.

In [ ]:
cheatsheet = """# MPNN Settings Cheat-Sheet (Project 02 — by <your name>, <date>)

Read off the Pareto front in notebook 04. NO setting is universally best; pick by goal.
Numbers below are PLACEHOLDERS — fill from results/settings_summary.csv (and label the dataset).

| Goal | Recommended temp | Recommended noise | seqs/backbone | Why (from your Pareto map) |
|------|------------------|-------------------|---------------|----------------------------|
| Max foldability (safe, conservative)   | ~0.1 | 0.0      | 8–16 | highest recapitulation rate; low diversity |
| Balanced (default for most pipelines)  | ~0.2 | 0.0–0.1  | 16   | good recap rate, usable diversity, OK solubility |
| Max diversity (give the filter choices)| ~0.3–0.5 | 0.1–0.2 | 48 | high entropy; expect lower recap rate — over-generate then filter |
| Max predicted solubility               | ~0.1–0.2 | 0.0    | 16–48 | best camsol_like proxy + low hydrophobic-patch fraction |

## Honest caveats (state these every time)
- Recapitulation (scRMSD < 2 A) is self-consistency, NOT proof of folding/stability/expression.
- net_charge / hydrophobic_fraction / camsol_like are PROXIES. camsol_like is a heuristic, NOT real
  CamSol (Sormanni 2015). Only express -> SDS-PAGE -> SEC measures expression.
- Cutoffs are calibrated on <your backbone set>; revisit for different fold classes / lengths.
- "diversity before filtering": prefer over-generating at higher temp and filtering hard over
  polishing one low-temp sequence.
"""
open("../data/CHEATSHEET.md", "w").write(cheatsheet)
print("wrote ../data/CHEATSHEET.md — fill the placeholders from notebook 04.")

## 2 · Settings-recommendation tool `[extension]`

A tiny heuristic that, given a backbone descriptor and a goal, returns a recommended setting from
your Pareto front. Wire it to `results/settings_summary.csv` for the real version; this stub encodes
the cheat-sheet logic so downstream students can import it.

In [ ]:
def recommend_settings(goal="balanced", length=120, ss_class="mixed"):
    """Recommend (temperature, noise, n_seqs) for a goal. Replace the table with your Pareto front.

    goal in {"foldability", "balanced", "diversity", "solubility"}.
    On the real version, load results/settings_summary.csv and pick the Pareto-optimal cell that
    maximizes the chosen objective subject to a minimum recapitulation rate.
    """
    table = {
        "foldability": dict(temperature=0.1, noise=0.0, n_seqs=16),
        "balanced":    dict(temperature=0.2, noise=0.1, n_seqs=16),
        "diversity":   dict(temperature=0.5, noise=0.2, n_seqs=48),
        "solubility":  dict(temperature=0.1, noise=0.0, n_seqs=48),
    }
    if goal not in table:
        raise ValueError(f"goal must be one of {sorted(table)}")
    rec = dict(table[goal]); rec["goal"] = goal
    return rec

for g in ("foldability", "balanced", "diversity", "solubility"):
    print(g, "->", recommend_settings(goal=g))

## 3 · Expression strategy + validation plan (D4)

The plan a wet lab would follow to test your recommended settings. **Controls are mandatory.**

In [ ]:
plan = """# Expression + Validation Plan (Project 02)

## Codon optimization & construct
- Optimize chosen sequences for the host's codon usage (e.g., E. coli); avoid rare-codon clusters
  and strong mRNA secondary structure near the start codon.
- N-terminal His6 tag with a cleavable linker (TEV) for IMAC purification; keep the tag out of any
  predicted hydrophobic patch.
- Order genes ONLY through a biosecurity-screening provider (IGSC member).

## Expression
- Host: E. coli BL21(DE3). Induce at 16-18 C overnight (low temperature favors soluble de novo
  monomer expression). Note when a different host is needed.

## Characterization tiers
1. Go/no-go: express -> SDS-PAGE (soluble vs inclusion bodies) -> SEC (monodisperse? right size?).
2. Basic: DSF (Tm) and/or CD (secondary structure matches the design?).
3. Deep (top picks): structure (X-ray/cryo-EM) or SEC-MALS/SAXS.

## Controls (MANDATORY)
- Positive control: a KNOWN-GOOD NATURAL monomer sequence (expresses + folds well).
- Negative control: a DELIBERATELY HIGH-HYDROPHOBIC-PATCH design (expected to express poorly /
  aggregate) — tests that the solubility proxy points the right way.
- Unrelated-protein control.

## Reporting
- Report the EXPRESSION HIT RATE per setting (N soluble / N tried), not the cherry. Include failures.
- Never present in-silico proxy scores as expression results.
"""
open("../data/PLAN.md", "w").write(plan)
print("wrote ../data/PLAN.md")

## 4 · (Stretch) MPNNsol / surface-redesign comparison `[stretch]`

Redesign **only the surface positions** for solubility (an MPNNsol-style pass that fixes the core and
lets surface residues vary toward soluble identities), then re-score: did the solubility proxies
improve without hurting recapitulation? Scaffold below — implement with real MPNN fixed-position
masks on Colab.

In [ ]:
# Scaffold (implement on Colab):
# 1. Compute per-residue burial (SASA from the backbone) -> classify core vs surface.
# 2. Re-run ProteinMPNN with --fixed_positions_jsonl pinning the CORE; let SURFACE vary.
# 3. Re-recapitulate + re-score net_charge / hydrophobic_fraction / camsol_like.
# 4. Compare surface-redesigned vs original: solubility proxy delta vs recapitulation delta.
print("MPNNsol surface-redesign scaffold — fix core, vary surface, re-score solubility proxies.")

## D4 / D5 checklist
- [ ] `data/CHEATSHEET.md` completed from the Pareto front, by goal, with honest caveats (D★).
- [ ] `recommend_settings(...)` wired to `results/settings_summary.csv`.
- [ ] `data/PLAN.md`: codon/tag + express→SDS-PAGE→SEC→DSF/CD + the three controls (incl. the
      high-hydrophobic-patch negative) + timeline + costed reagents.
- [ ] (Stretch) MPNNsol surface-redesign comparison.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — and the cohort's MPNN-running projects now have your settings cheat-sheet.